https://platform.olimpiada-ai.ro/ro/problems/51

# setup + imports

In [1]:
import pandas as pd 
import numpy as np

In [2]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from sklearn.dummy import DummyRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import OrdinalEncoder

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [3]:
train_data = pd.read_csv('train_data.csv')
test_data = pd.read_csv('test_data.csv')

# subtask 1

In [4]:
answer1 = 0
for i in range(len(test_data)):
    if test_data['Sex'].iloc[i] == 'Femelă' and test_data['Culoare'].iloc[i] == 'Havana' and test_data['Ureche_lăsată'].iloc[i] == True:
        answer1 += 1

print(answer1)

11


# data exploration

In [5]:
# missing values?
print(train_data.isna().sum())

ID                0
Sex               0
Greutate          0
Lungime_urechi    0
Ureche_lăsată     0
Culoare           0
Vârstă            0
Tip_blană         0
Calitate_blană    0
Formă_corp        0
Apare_gușa        0
Sănătate          0
Scor_jurizare     0
dtype: int64


In [6]:
print(train_data["Culoare"].unique())
print(train_data["Tip_blană"].unique())
print(train_data["Calitate_blană"].unique())
print(train_data["Formă_corp"].unique())
print(train_data["Sănătate"].unique())

<StringArray>
['Havana', 'Agouti', 'Alb', 'Albastru', 'Negru']
Length: 5, dtype: str
<StringArray>
['Scurtă', 'Lungă']
Length: 2, dtype: str
<StringArray>
['Excelentă', 'Foarte_bună', 'Bună']
Length: 3, dtype: str
<StringArray>
['Incorectă', 'Aproape_corectă', 'Corectă']
Length: 3, dtype: str
<StringArray>
[ 'Probleme_grave_de_sănătate', 'Ușoare_probleme_de_sănătate',
           'Aspect_neîngrijit',                     'Sănătos']
Length: 4, dtype: str


# data preprocessing

In [7]:
def OH(data, col, categs):
    data = data.copy()

    for c in categs:
        newcol = []
        for i in range(len(data)):
            if data[col].iloc[i] == c:
                newcol.append(1)
            else:
                newcol.append(0)
        data[str(col+'_'+c)] = newcol
    data = data.drop(columns=col)
    
    return data

In [8]:
def OE(data, col, order):
    data = data.copy()

    newcol = []
    for c in data[col]:
        newcol.append(order[c])
    data = data.drop(columns=col)
    data[col] = newcol

    return data

In [9]:
def upgrade(data):
    data = data.copy()

    data = OH(data, 'Sex', ['Femelă', 'Mascul'])
    data = OH(data, 'Culoare', ['Havana', 'Agouti', 'Alb', 'Albastru','Negru'])
    data = OH(data, 'Tip_blană', ['Scurtă', 'Lungă'])

    data = OE(data, 'Calitate_blană', {'Excelentă':2, 'Foarte_bună':1, 'Bună':0})
    data = OE(data, 'Formă_corp', {'Incorectă':0, 'Aproape_corectă':1, 'Corectă':2})
    data = OE(data, 'Sănătate', {'Probleme_grave_de_sănătate':3, 'Ușoare_probleme_de_sănătate':2,'Aspect_neîngrijit':1, 'Sănătos':0})    

    data = OE(data, 'Ureche_lăsată', {True:0, False:1})
    data = OE(data, 'Apare_gușa', {True:0, False:1})

    return data

In [10]:
train_data = upgrade(train_data)
test_data = upgrade(test_data)


X = train_data.drop(columns=['Scor_jurizare', 'ID'])
cols = X.columns
y = train_data['Scor_jurizare']

In [11]:
print(train_data.dtypes)
print(cols)

ID                    int64
Greutate            float64
Lungime_urechi      float64
Vârstă                int64
Scor_jurizare         int64
Sex_Femelă            int64
Sex_Mascul            int64
Culoare_Havana        int64
Culoare_Agouti        int64
Culoare_Alb           int64
Culoare_Albastru      int64
Culoare_Negru         int64
Tip_blană_Scurtă      int64
Tip_blană_Lungă       int64
Calitate_blană        int64
Formă_corp            int64
Sănătate              int64
Ureche_lăsată         int64
Apare_gușa            int64
dtype: object
Index(['Greutate', 'Lungime_urechi', 'Vârstă', 'Sex_Femelă', 'Sex_Mascul',
       'Culoare_Havana', 'Culoare_Agouti', 'Culoare_Alb', 'Culoare_Albastru',
       'Culoare_Negru', 'Tip_blană_Scurtă', 'Tip_blană_Lungă',
       'Calitate_blană', 'Formă_corp', 'Sănătate', 'Ureche_lăsată',
       'Apare_gușa'],
      dtype='str')


# subtask 2

## data processing

In [12]:
scaler = StandardScaler()

XX = scaler.fit_transform(X)
X_test = test_data[cols]
X_test = scaler.transform(X_test)

## model

In [13]:
km = KMeans(n_clusters=3 , random_state=42 , n_init =20 , max_iter=500)

km.fit(XX)

clusters = km.predict(X_test)

# subtask 3

## validare

In [14]:
def scoremetric(preds, y_val):
    return mean_squared_error(preds, y_val)

def evaluate(model, X, y):
    kf = KFold(n_splits=5, random_state=0, shuffle=True)

    scores = []
    for fit_rows, val_rows in kf.split(X, y):
        X_fit = X.iloc[fit_rows]
        y_fit = y.iloc[fit_rows]
        X_val = X.iloc[val_rows]
        y_val = y.iloc[val_rows]

        model.fit(X_fit, y_fit)
        preds = model.predict(X_val)
        scores.append(scoremetric(preds, y_val))

    score = np.mean(scores)
    print('mean:', round(np.mean(scores), 4), ' spread:', round(np.std(scores), 4))
    

# model

In [15]:
model = HistGradientBoostingRegressor()
evaluate(model, X, y)

mean: 1.9024  spread: 0.4774


In [16]:
model.fit(X, y)
preds = model.predict(test_data[cols])

# submisie

In [17]:
subtask1 = pd.DataFrame({
    'subtaskID': 1,
    'datapointID': 1,
    'answer': answer1
}, index=[0])


subtask2 = pd.DataFrame({
    'subtaskID': 2,
    'datapointID': test_data['ID'],
    'answer': clusters
})

subtask3 = pd.DataFrame({
    'subtaskID': 3,
    'datapointID': test_data['ID'],
    'answer': preds
})
     

submission = pd.concat([subtask1, subtask2, subtask3], ignore_index=True)
submission.to_csv('submission.csv', index=False)
     